# M38 lab solution: a retriever over MCP

**Brief.** Retriever over MCP: expose your Part V retriever as an MCP tool and call it from an MCP client chatbot.

- [`server.py`](server.py) is a FastMCP server, `larkspur-handbook`, with one tool, `search_handbook(query, k)`. Its retriever is the lab baseline (recursive split plus TF-IDF from `src/lab_kit.py`), rebuilt inline so the lab depends on no other lab. It runs offline.
- This notebook is the MCP client. It starts the server as a subprocess over stdio, lists its tools, and runs a chatbot loop: the model sees the tool's JSON schema, asks for calls, the client forwards each call to the server with `session.call_tool`, and the result goes back to the model as a tool message. This is the loop of notebook `03a_creating_an_mcp_client.ipynb`, with a LangChain chat model in place of the OpenAI client.

## Setup

In [ ]:
import asyncio
import os
import sys
import time
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
nest_asyncio.apply()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import load_jsonl  # noqa: E402

golden = {g["id"]: g for g in load_jsonl(Path("../data/larkspur/golden.jsonl"))}
QUESTIONS = ["g01", "g04", "g10"]
llm = get_experientiallabs_llm()
MODEL = llm.model_name

SERVER = StdioServerParameters(command=sys.executable, args=["server.py"], cwd=str(Path(".").resolve()))
print("model:", MODEL, "| server:", SERVER.command, SERVER.args)

## Step 1: list the server's tools and call one directly

No model is involved here: this checks the server end to end. `errlog` goes to `os.devnull` because a Jupyter kernel's `sys.stderr` has no file descriptor for the subprocess to inherit.

In [ ]:
async def inspect_server() -> None:
    with open(os.devnull, "w") as errlog:
        async with stdio_client(SERVER, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = (await session.list_tools()).tools
                for t in tools:
                    print(t.name, "-", t.description)
                    print("  schema:", t.inputSchema)
                result = await session.call_tool("search_handbook", arguments={"query": golden["g01"]["question"], "k": 2})
                print(result.content[0].text)


asyncio.run(inspect_server())

## Step 2: the chatbot loop

The MCP tool list is turned into OpenAI-style tool definitions for `bind_tools`. The loop runs until the model answers without asking for a tool, with a cap of four tool rounds.

In [ ]:
SYSTEM = ("You are Larkspur Cloud's support assistant. Use the search_handbook tool to find the answer, "
          "answer only from the passages it returns, and name the source file in brackets.")
MAX_TOOL_ROUNDS = 4


async def chat(questions: list[str]) -> list[dict]:
    out = []
    with open(os.devnull, "w") as errlog:
        async with stdio_client(SERVER, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = (await session.list_tools()).tools
                bound = llm.bind_tools([
                    {"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.inputSchema}}
                    for t in tools
                ])
                for q in questions:
                    messages = [SystemMessage(SYSTEM), HumanMessage(q)]
                    calls = []
                    start = time.perf_counter()
                    for _ in range(MAX_TOOL_ROUNDS):
                        reply = await bound.ainvoke(messages)
                        messages.append(reply)
                        if not reply.tool_calls:
                            break
                        for call in reply.tool_calls:
                            result = await session.call_tool(call["name"], arguments=call["args"])
                            text = "\n".join(c.text for c in result.content if getattr(c, "text", None))
                            calls.append({"tool": call["name"], "args": call["args"]})
                            messages.append(ToolMessage(content=text, tool_call_id=call["id"]))
                    out.append({"question": q, "tool_calls": calls, "answer": messages[-1].content,
                                "seconds": round(time.perf_counter() - start, 2)})
    return out


answers = asyncio.run(chat([golden[i]["question"] for i in QUESTIONS]))
for gid, a in zip(QUESTIONS, answers):
    print(f"{gid}: {a['question']}")
    print("  tool calls:", a["tool_calls"])
    print("  answer:", a["answer"])
    print("  reference:", golden[gid]["reference_answer"])

## Results

`spans_in_answer` is a rough check, not a grade: whether any evidence span from the golden set appears in the answer after lower-casing. Read the answers above against the references.

In [ ]:
import pandas as pd
from lab_kit import normalise

rows = []
for gid, a in zip(QUESTIONS, answers):
    spans = golden[gid]["evidence_spans"]
    rows.append({"id": gid, "tool_calls": len(a["tool_calls"]), "seconds": a["seconds"],
                 "spans_in_answer": any(normalise(s) in normalise(a["answer"]) for s in spans)})
print("model:", MODEL, "| run:", time.strftime("%Y-%m-%d"))
print(pd.DataFrame(rows).to_markdown(index=False))

## What this shows

- The retriever now lives behind a protocol: any MCP client that speaks stdio can list `search_handbook`, read its schema and call it, without importing the retriever's code.
- The client does the work the model cannot: it holds the session, forwards each tool call and returns the result. The model only sees a tool schema and tool messages.
- The server needs no keys and makes no model calls, so a failure here is either the transport (Step 1 fails) or the model's tool use (Step 1 passes, Step 2 does not).